In [ ]:
import sys 
sys.path.append('/host/d/Github')
import os
import torch
import numpy as np 
import Diffusion_denoising_thin_slice.Thinslice_experiments.denoising_diffusion_pytorch.denoising_diffusion_pytorch.conditional_diffusion as ddpm
import Diffusion_denoising_thin_slice.functions_collection as ff
import Diffusion_denoising_thin_slice.Build_lists.Build_list as Build_list
import Diffusion_denoising_thin_slice.Generator_thinslice as Generator

trial_name = 'supervised_gaussian_brainCT' 
problem_dimension = '2D'
supervision = 'supervised' if trial_name[0:2] == 'su' else 'unsupervised'; print('supervision:', supervision)

# bias  
beta = 0
lpips_weight = 0#0.2
edge_weight = 0#0.05

# model condition 
# if 'mean' in trial_name: condition on current slice, target the mean of neighboring slices
# else: condition on neighboring slices, target the current slice
condition_channel = 1 if (supervision == 'supervised') or ('mean' in trial_name) else 2

pre_trained_model = None#os.path.join('/host/d/projects/denoising/models', trial_name, 'models/model-2640.pt') #None
start_step = 0#2640
image_size = [512,512]
num_patches_per_slice = 2
patch_size = [128,128]

objective = 'pred_x0'

histogram_equalization = True
background_cutoff = -1000
maximum_cutoff = 2000
normalize_factor = 'equation'

###########################
# define train
if supervision == 'supervised':
    build_sheet =  Build_list.Build_thinsliceCT(os.path.join('/host/d/Data/brain_CT/Patient_lists/fixedCT_static_simulation_train_test_gaussian_xjtlu.xlsx'))
else:
    build_sheet =  Build_list.Build_thinsliceCT(os.path.join('/host/d/Data/brain_CT/Patient_lists/fixedCT_static_simulation_train_test_gaussian_xjtlu.xlsx'))

_,_,_,_, condition_list_train, x0_list_train = build_sheet.__build__(batch_list = [0,1,2,3]) 
# n = ff.get_X_numbers_in_interval(total_number = x0_list_train.shape[0],start_number = 1,end_number = 2, interval = 2)
# x0_list_train = x0_list_train[n]; condition_list_train = condition_list_train[n]
# x0_list_train = x0_list_train[0:1]; condition_list_train = condition_list_train[0:1]

# define val
_,_,_,_, condition_list_val, x0_list_val = build_sheet.__build__(batch_list = [4,5])
# n = ff.get_X_numbers_in_interval(total_number = x0_list_val.shape[0],start_number = 1,end_number = 2, interval = 2)
# x0_list_val = x0_list_val[n]; condition_list_val = condition_list_val[n]
# x0_list_val = x0_list_val[0:1]; condition_list_val = condition_list_val[0:1]

print('train:', x0_list_train.shape, condition_list_train.shape, 'val:', x0_list_val.shape, condition_list_val.shape)
print(x0_list_train[0:5], condition_list_train[0:5], x0_list_val[0:5], condition_list_val[0:5])


/opt/conda/lib/python3.10/site-packages/torchvision/io/image.py:13: UserWarning: Failed to load image Python extension: 'Could not load this library: /opt/conda/lib/python3.10/site-packages/torchvision/image.so'If you don't plan on using image functionality from `torchvision.io`, you can ignore this warning. Otherwise, there might be something wrong with your environment. Did you have `libjpeg` or `libpng` installed before building `torchvision` from source?
  warn(
/opt/conda/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


supervision: supervised
train: (68,) (68,) val: (32,) (32,)
['/host/d/Data/brain_CT/fixedCT/00139437/0000258390/img_thinslice_partial.nii.gz'
 '/host/d/Data/brain_CT/fixedCT/00019599/0000029506/img_thinslice_partial.nii.gz'
 '/host/d/Data/brain_CT/fixedCT/00148611/0000455363/img_thinslice_partial.nii.gz'
 '/host/d/Data/brain_CT/fixedCT/00173704/0000455358/img_thinslice_partial.nii.gz'
 '/host/d/Data/brain_CT/fixedCT/00030597/0000455473/img_thinslice_partial.nii.gz'] ['/host/d/Data/brain_CT/simulation/00139437/0000258390/gaussian_random_1/recon.nii.gz'
 '/host/d/Data/brain_CT/simulation/00019599/0000029506/gaussian_random_1/recon.nii.gz'
 '/host/d/Data/brain_CT/simulation/00148611/0000455363/gaussian_random_1/recon.nii.gz'
 '/host/d/Data/brain_CT/simulation/00173704/0000455358/gaussian_random_1/recon.nii.gz'
 '/host/d/Data/brain_CT/simulation/00030597/0000455473/gaussian_random_1/recon.nii.gz'] ['/host/d/Data/brain_CT/fixedCT/00134441/0000455662/img_thinslice_partial.nii.gz'
 '/host/d/D

/host/d/Github/Diffusion_denoising_thin_slice/denoising_diffusion_pytorch/denoising_diffusion_pytorch/conditional_diffusion.py:1001: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  @autocast(enabled = False)
/host/d/Github/Diffusion_denoising_thin_slice/Thinslice_experiments/denoising_diffusion_pytorch/denoising_diffusion_pytorch/conditional_diffusion.py:1001: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  @autocast(enabled = False)


In [2]:
import nibabel as nb
a = [x0_list_train, condition_list_train, x0_list_val, condition_list_val]

for aa in a:
    for file in aa:
        f = nb.load(file).get_fdata()
        shape = f.shape
        for slice_n in range(shape[2]):
            slice_data = f[:,:,slice_n]
            # assess whether any pixel is np.nan or sum is np.nan
            if np.isnan(slice_data).any() or np.isnan(slice_data.sum()):
                print(f"NaN detected in file: {file}, slice: {slice_n}")

/opt/conda/lib/python3.10/site-packages/numpy/core/_methods.py:49: RuntimeWarning: overflow encountered in reduce
  return umr_sum(a, axis, dtype, out, keepdims, initial, where)


NaN detected in file: /host/d/Data/brain_CT/fixedCT/00010536/0000455747/img_thinslice_partial.nii.gz, slice: 20
NaN detected in file: /host/d/Data/brain_CT/fixedCT/00010536/0000455747/img_thinslice_partial.nii.gz, slice: 21
NaN detected in file: /host/d/Data/brain_CT/fixedCT/00010536/0000455747/img_thinslice_partial.nii.gz, slice: 22
NaN detected in file: /host/d/Data/brain_CT/fixedCT/00010536/0000455747/img_thinslice_partial.nii.gz, slice: 23
NaN detected in file: /host/d/Data/brain_CT/fixedCT/00010536/0000455747/img_thinslice_partial.nii.gz, slice: 24
NaN detected in file: /host/d/Data/brain_CT/fixedCT/00010536/0000455747/img_thinslice_partial.nii.gz, slice: 25
NaN detected in file: /host/d/Data/brain_CT/fixedCT/00010536/0000455747/img_thinslice_partial.nii.gz, slice: 26
NaN detected in file: /host/d/Data/brain_CT/fixedCT/00010536/0000455747/img_thinslice_partial.nii.gz, slice: 27
NaN detected in file: /host/d/Data/brain_CT/fixedCT/00010536/0000455747/img_thinslice_partial.nii.gz, sl

/opt/conda/lib/python3.10/site-packages/nibabel/arrayproxy.py:408: RuntimeWarning: invalid value encountered in cast
  scaled = scaled.astype(np.promote_types(scaled.dtype, dtype), copy=False)


NaN detected in file: /host/d/Data/brain_CT/simulation/00139437/0000258390/gaussian_random_1/recon.nii.gz, slice: 25
NaN detected in file: /host/d/Data/brain_CT/simulation/00139437/0000258390/gaussian_random_1/recon.nii.gz, slice: 26
NaN detected in file: /host/d/Data/brain_CT/simulation/00139437/0000258390/gaussian_random_1/recon.nii.gz, slice: 27
NaN detected in file: /host/d/Data/brain_CT/simulation/00139437/0000258390/gaussian_random_1/recon.nii.gz, slice: 28
NaN detected in file: /host/d/Data/brain_CT/simulation/00139437/0000258390/gaussian_random_1/recon.nii.gz, slice: 29
NaN detected in file: /host/d/Data/brain_CT/simulation/00139437/0000258390/gaussian_random_1/recon.nii.gz, slice: 30
NaN detected in file: /host/d/Data/brain_CT/simulation/00139437/0000258390/gaussian_random_1/recon.nii.gz, slice: 31
NaN detected in file: /host/d/Data/brain_CT/simulation/00139437/0000258390/gaussian_random_1/recon.nii.gz, slice: 32
NaN detected in file: /host/d/Data/brain_CT/simulation/00139437/